# Câu C 

## TIỀN XỬ LÝ DỮ LIỆU

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import underthesea   
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [26]:
Path = os.path.join(os.getcwd(), "Data")


def read_file(filename):
    filename = os.path.join(Path, filename)
    with open(filename, 'r', encoding="utf-16") as f:
        data = f.read()
    return data


def natural_key(name):
    m = re.match(r"(.+?)_ \((\d+)\)\.txt$", name)
    return (m.group(1), int(m.group(2))) if m else (name, 0)

file_names = sorted([f for f in os.listdir(Path) if f.endswith(".txt")], key=natural_key)

Raw_Data = {}      
errors = []
for name in file_names:
    try:
        Raw_Data[name] = read_file(name)
    except Exception as e:
        errors.append((name, str(e)))

print(f"Đã đọc: {len(Raw_Data)} file | Lỗi: {len(errors)}")
for name, err in errors:
    print("  ✗", name, "-", err)

Đã đọc: 475 file | Lỗi: 0


In [27]:
StopWords_File = os.path.join(os.getcwd(), "vietnamese-stopwords.txt")

if os.path.exists(StopWords_File):
    with open(StopWords_File, "r", encoding="utf-8") as f:
        List_StopWords = set(w.strip().lower().replace(" ", "_")
                             for w in f.read().split("\n") if w.strip())
   
else:
    List_StopWords = set()
   

In [28]:
def Text_Preprocessing(doc):
    text_pre = doc.lower()                                   # Chuyển chữ thường

    text_pre = underthesea.word_tokenize(text_pre, format="text")   # Tách từ tiếng Việt
    text_pre = underthesea.text_normalize(text_pre)                 # Chuẩn hóa dấu tiếng Việt

    text_pre = text_pre.replace(u'�', '')               # Xóa ký tự lỗi
    text_pre = re.sub(r'http\S+', '', text_pre)              # Xóa URL có http
    text_pre = re.sub(r'www\S+', '', text_pre)               # Xóa URL có www
    text_pre = re.sub(r"\S*@\S*\s?", " ", text_pre)          # Xóa email
    text_pre = re.sub(r'[^\w\s]', ' ', text_pre)             # Xóa dấu câu, ký tự đặc biệt (giữ dấu _)
    text_pre = re.sub(r"\S*\d\S*", " ", text_pre)            # Xóa số và từ dính số
    text_pre = re.sub(r'\s+', ' ', text_pre).strip()         # Xóa khoảng trắng thừa, xuống dòng

    # Loại bỏ stopwords
    if List_StopWords:
        text_pre = " ".join(w for w in text_pre.split() if w not in List_StopWords)

    return text_pre

In [29]:
Data = {}         
for name, content in Raw_Data.items():
    Data[name] = Text_Preprocessing(content)


empty_docs = [n for n, t in Data.items() if not t.strip()]
for n in empty_docs:
    del Data[n]
print(f"Số văn bản sau tiền xử lý: {len(Data)} ")


Số văn bản sau tiền xử lý: 475 


In [30]:
file_list = list(Data.keys())        
corpus    = list(Data.values())      

tfidf_vectorizer = TfidfVectorizer(max_df=0.5, min_df=2)
X = tfidf_vectorizer.fit_transform(corpus)
terms = tfidf_vectorizer.get_feature_names_out()
print("Kích thước ma trận TF-IDF:", X.shape )

Kích thước ma trận TF-IDF: (475, 4836)


## COSINE SIMILARITY

In [31]:
cosine_similarity_matrix = cosine_similarity(X)
print("Kích thước ma trận:", cosine_similarity_matrix.shape)

Similarity = pd.DataFrame(cosine_similarity_matrix, index=file_list, columns=file_list)

Kích thước ma trận: (475, 475)


## TOP 5 CẶP VĂN BẢN TƯƠNG ĐỒNG NHẤT

In [34]:
n = len(file_list)
i_idx, j_idx = np.triu_indices(n, k=1)          
scores = cosine_similarity_matrix[i_idx, j_idx]

pairs = pd.DataFrame({
    'Văn bản 1': [file_list[i] for i in i_idx],
    'Văn bản 2': [file_list[j] for j in j_idx],
    'Cosine'   : scores
}).sort_values('Cosine', ascending=False).reset_index(drop=True)

top5 = pairs.head(5)
print("TOP 5 TƯƠNG ĐỒNG CAO NHẤT:")
for k, row in top5.iterrows():
    print(f"{k+1}. {row['Văn bản 1']:<22} -  {row['Văn bản 2']:<22} : {row['Cosine']:.4f}")

TOP 5 TƯƠNG ĐỒNG CAO NHẤT:
1. DL_NLD_ (576).txt      -  DL_TT_T_ (336).txt     : 0.9175
2. DL_TN_ (301).txt       -  DL_TN_ (311).txt       : 0.8574
3. BDS_VNE_ (22).txt      -  BDS_VNE_ (59).txt      : 0.7811
4. BDS_VNE_ (21).txt      -  BDS_VNE_ (22).txt      : 0.7544
5. GDu_NLD_ (634).txt     -  GDu_NLD_ (638).txt     : 0.7497


In [35]:
best = top5.iloc[0]
for f in [best['Văn bản 1'], best['Văn bản 2']]:
    print(f"--- {f} ---")
    print(Raw_Data[f][:300].replace("\n", " "), "...\n")

n_dup = (pairs['Cosine'] > 0.95).sum()

--- DL_NLD_ (576).txt ---
 Hơn 16.000 khách đến vịnh Nha Trang Theo trực ban bộ đội biên phòng tại cảng du lịch Cầu Đá (Vĩnh Nguyên), trong những ngày lễ vừa qua có hơn 16.000 người đến tham quan vịnh Nha Trang, trong đó có 734 du khách nước ngoài. Đông nhất là vào ngày 30-4, vịnh Nha Trang đã đón tới 7.019 du khách. Những đ ...

--- DL_TT_T_ (336).txt ---
 Hơn 16.000 khách đến vịnh Nha Trang Đông nhất là vào ngày 30-4, vịnh Nha Trang đã đón tới 7.019 du khách. Những địa chỉ trong vịnh Nha Trang thu hút được số lượng du khách đến tham quan, nghỉ ngơi, vui chơi nhiều nhất vẫn là các đảo Hòn Tằm và Hòn Mun.    ...

